In [20]:
import tensorflow as tf
from tensorflow.keras import layers, models

# -----------------------------
# CONFIG
# -----------------------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

TRAIN_DIR = "../dataset_split/train"
VAL_DIR = "../dataset_split/val"

# -----------------------------
# LOAD DATA
# -----------------------------
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Normalize (0–1)
normalization_layer = layers.Rescaling(1./255) 
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)) 
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# -----------------------------
# LOAD PRETRAINED MODEL
# -----------------------------
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# -----------------------------
# BUILD MODEL
# -----------------------------
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

# -----------------------------
# COMPILE
# -----------------------------
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -----------------------------
# TRAIN
# -----------------------------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

# -----------------------------
# SAVE MODEL
# -----------------------------
model.save("model5.keras")

Found 1165 files belonging to 2 classes.


Found 293 files belonging to 2 classes.
Epoch 1/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 23s 526ms/step - accuracy: 0.7193 - loss: 0.5547 - val_accuracy: 0.7713 - val_loss: 0.4324
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 18s 484ms/step - accuracy: 0.8060 - loss: 0.4049 - val_accuracy: 0.7816 - val_loss: 0.4997
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 19s 507ms/step - accuracy: 0.8275 - loss: 0.3604 - val_accuracy: 0.8362 - val_loss: 0.3623
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 18s 497ms/step - accuracy: 0.8618 - loss: 0.3073 - val_accuracy: 0.8328 - val_loss: 0.3620
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 20s 537ms/step - accuracy: 0.8755 - loss: 0.2761 - val_accuracy: 0.8225 - val_loss: 0.4166
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 19s 516ms/step - accuracy: 0.8944 - loss: 0.2405 - val_accuracy: 0.8498 - val_loss: 0.3710
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 18s 497ms/step - accuracy: 0.8927 - loss: 0.2364 - val_accuracy: 0.8532 - val_loss: 0.3336
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 18s 494ms/step - ac

In [33]:
model = tf.keras.models.load_model("model.keras")

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "../dataset_split/val",
    image_size=(224,224),
    batch_size=32,
    shuffle=False   # IMPORTANT
)

val_ds = val_ds.map(lambda x, y: (x/255.0, y))

loss, acc = model.evaluate(val_ds)

print("Validation Accuracy:", acc)

Found 293 files belonging to 2 classes.
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 387ms/step - accuracy: 0.8498 - loss: 0.3659
Validation Accuracy: 0.849829375743866


In [34]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images)
    preds = (preds > 0.5).astype(int)

    y_true.extend(labels.numpy())
    y_pred.extend(preds.flatten())

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 412ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 448ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 410ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 458ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 410ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 805ms/step
[[164  31]
 [ 13  85]]
              precision    recall  f1-score   support

           0       0.93      0.84      0.88       195
           1       0.73      0.87      0.79        98

    accuracy                           0.85       293
   macro avg       0.83      0.85      0.84       293
weighted avg       0.86      0.85      0.85       293

